# NB159 — Cascade Eigenvalue Sweep

**Context**: The scaling factors r_bs=1.269, r_tc=1.639 are eigenvalues of the
coupled cascade that we cannot derive analytically. NB158 showed the R0 analytic
formula alone is insufficient.

**Strategy**: Vary kappa continuously (decoupled from 1/sqrt(P4)) while keeping
the CRT structure fixed. Measure how x_q, r_bs, r_tc, and the CP ratios depend
on kappa. If these follow a formula, the sweep will reveal it.

**Using**: JAX backend for ~200x speedup (see docs/acceleration.md).

In [2]:
# S0 — Kappa sweep: how do exponents depend on the spatial decay rate?
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
omega = 2 * np.pi
kappa_natural = 1.0 / np.sqrt(P4)  # 0.06901

cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

# CP pair indices
def get_cp_indices():
    idx = {}
    for name, (ch_a3, a7_g1, a7_g2) in CP_PAIRS.items():
        g1_mask = (a3 == ch_a3) & (a5 == 0) & (a7 == a7_g1)
        g2_mask = (a3 == ch_a3) & (a5 == 0) & (a7 == a7_g2)
        idx[name] = (np.where(g1_mask)[0][0], np.where(g2_mask)[0][0])
    return idx

cp_idx = get_cp_indices()

from solenoid_jax import warmup
warmup()

def run_cascade_at_kappa(kappa_val):
    """Run cascade with specified kappa, return CP ratios at all levels."""
    sys0 = SolenoidSystem(primes=primes, kappa=kappa_val, epsilon=kappa_val)
    branches = sys0.all_branches()
    t_eval = cis.astype(float)
    T_max = float(P4 + 1)
    res = sys0.integrate_all_branches(branches, t_eval, T_max, backend='jax')
    
    # Compute RMS at all crossings
    n_cross = len(cis)
    rms = np.zeros((n_cross, 4))
    for idx in range(n_cross):
        for k in range(4):
            Rk = np.array([res[br][idx, k] for br in branches])
            Rk_w = np.mod(Rk, 2 * np.pi)
            Rk_w[Rk_w > np.pi] -= 2 * np.pi
            rms[idx, k] = np.sqrt(np.mean(Rk_w**2))
    
    # CP ratios
    cp = {}
    for name in ['QUARK', 'LEPTON']:
        i1, i2 = cp_idx[name]
        cp[name] = rms[i1] / rms[i2]
    
    return cp, rms

# ══════════════════════════════════════════════════════════════
# Sweep kappa from 0.5× to 2× the natural value
# ══════════════════════════════════════════════════════════════

kappa_factors = np.array([0.3, 0.5, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.5, 2.0, 3.0])
kappa_values = kappa_natural * kappa_factors

print(f'=== Kappa sweep: {len(kappa_values)} values ===')
print(f'Natural kappa = 1/sqrt({P4}) = {kappa_natural:.6f}')
print(f'Sweep range: {kappa_values[0]:.6f} to {kappa_values[-1]:.6f}')

results = []
for i, kf in enumerate(kappa_factors):
    kv = kappa_natural * kf
    try:
        cp, rms = run_cascade_at_kappa(kv)
        # Compute x_q from the quark CP at R3
        cp_R3_q = cp['QUARK'][3]
        cp_R3_l = cp['LEPTON'][3]
        
        # PDG ratios (these don't change with kappa)
        m_sd = 20.0  # m_s/m_d
        m_bs = 44.75
        m_tc = 135.8
        m_me = 206.768
        
        # Exponents: x = ln(mass_ratio) / ln(CP)
        x_q = np.log(m_sd) / np.log(cp_R3_q) if cp_R3_q > 1.01 else np.nan
        x_l = np.log(m_me) / np.log(cp_R3_l) if cp_R3_l > 1.01 else np.nan
        
        # Scaling factors from PDG
        r_bs = np.log(m_bs) / np.log(m_sd) if m_sd > 1.01 else np.nan
        r_tc = np.log(m_tc) / np.log(m_sd) if m_sd > 1.01 else np.nan
        # Note: r_bs and r_tc are PDG-derived, they DON'T change with kappa!
        # What DOES change is the CP ratio.
        # The question is: does CP^{x_q} still give 20.0 at different kappa?
        # i.e., does x_q adjust to keep CP^x = 20?
        # No — x_q is defined as ln(20)/ln(CP), so it ALWAYS gives 20.
        # The real question: is x_q close to a clean fraction for other kappas?
        
        # Compute x_q at R0 (the analytic level)
        cp_R0_q = cp['QUARK'][0]
        x_q_R0 = np.log(m_sd) / np.log(cp_R0_q) if cp_R0_q > 1.01 else np.nan
        
        # Cross-level
        cl = np.log(cp_R0_q) / np.log(cp_R3_q) if cp_R3_q > 1.01 else np.nan
        
        results.append({
            'kf': kf, 'kappa': kv,
            'cp_R3_q': cp_R3_q, 'cp_R0_q': cp_R0_q,
            'cp_R3_l': cp_R3_l,
            'x_q': x_q, 'x_l': x_l,
            'x_q_R0': x_q_R0, 'cl': cl,
        })
        print(f'  kf={kf:.1f}: CP_R3(Q)={cp_R3_q:.4f} x_q={x_q:.6f} x_q(R0)={x_q_R0:.6f} cl={cl:.4f}')
    except Exception as e:
        print(f'  kf={kf:.1f}: FAILED ({e})')

# ══════════════════════════════════════════════════════════════
# Analysis: how does x_q depend on kappa?
# ══════════════════════════════════════════════════════════════

print(f'\n=== x_q(R0) vs kappa ===')
print(f'{"kf":>5} {"kappa":>10} {"x_q(R0)":>12} {"near 4/7?":>10} {"x_q(R3)":>12} {"CP_R3":>10}')
for r in results:
    near_47 = abs(r['x_q_R0'] - 4/7) / (4/7) * 100 if not np.isnan(r['x_q_R0']) else 999
    print(f'{r["kf"]:5.1f} {r["kappa"]:10.6f} {r["x_q_R0"]:12.6f} {near_47:9.3f}% {r["x_q"]:12.6f} {r["cp_R3_q"]:10.4f}')

# KEY QUESTION: Is x_q(R0) = 4/7 ONLY at the natural kappa = 1/sqrt(P4)?
# Or does it hold at all kappas?
# If it's ONLY at the natural kappa, then 4/7 comes from the 
# specific relationship kappa = 1/sqrt(P4).
# If it holds at all kappas, then 4/7 is structural (from the primes/CRT).

print(f'\n=== Is x_q(R0) = 4/7 specific to kappa = 1/sqrt(P4)? ===')
natural_x_R0 = [r['x_q_R0'] for r in results if r['kf'] == 1.0][0]
other_x_R0 = [(r['kf'], r['x_q_R0']) for r in results if r['kf'] != 1.0 and not np.isnan(r['x_q_R0'])]
spread = max(abs(x - natural_x_R0) for _, x in other_x_R0) if other_x_R0 else 0
print(f'  Natural (kf=1.0): x_q(R0) = {natural_x_R0:.8f}')
print(f'  Max deviation from natural: {spread:.6f} ({spread/natural_x_R0*100:.2f}%)')
if spread < 0.001:
    print(f'  → x_q(R0) is KAPPA-INDEPENDENT! It comes from the primes/CRT, not from kappa.')
else:
    print(f'  → x_q(R0) DEPENDS on kappa. It comes from the cascade dynamics.')
    print(f'  Variation with kappa:')
    for kf, x in other_x_R0:
        print(f'    kf={kf:.1f}: x_q(R0) = {x:.8f} (dev from 4/7: {abs(x-4/7)/(4/7)*100:.3f}%)')

=== Kappa sweep: 11 values ===
Natural kappa = 1/sqrt(210) = 0.069007
Sweep range: 0.020702 to 0.207020
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.52s
  kf=0.3: CP_R3(Q)=2.0345 x_q=4.217835 x_q(R0)=1.253405 cl=3.3651
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.24s
  kf=0.5: CP_R3(Q)=8.8989 x_q=1.370465 x_q(R0)=0.521025 cl=2.6303
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.36s
  kf=0.7: CP_R3(Q)=8.9223 x_q=1.368820 x_q(R0)=0.543227 cl=2.5198
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.45s
  kf=0.8: CP_R3(Q)=7.7270 x_q=1.465107 x_q(R0)=0.549707 cl=2.6652
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.39s
  kf=0.9: CP_R3(Q)=6.5783 x_q=1.590282 x_q(R0)=0.553864 cl=2.8712
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.52s
  kf=1.0: CP_R3(Q)=6.6067 x_q=1.586646 x_q(R0)=0.571450 cl=2.7765
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.36s
  kf=1.1: 